# Phase 1, test P1b: were the gray-lift draws identical to single-threaded draws? (CPU only)

`gray-lift-kill-test` generated the gray noise with NumPy's *global* RNG (`np.random.seed(seed)` then
`np.random.normal`) inside two GPU worker threads, so a draw can interleave with the other thread's. The Xformer,
GRL-B, SCUNet and DRUNet reruns drew single-threaded. Restormer-gray ran inside the lift job on the lift's own draws,
so rerunning it here on single-threaded draws reproduces its per-image PSNR iff the draws were identical.

Pre-registered (decision log, Phase 1): same draw ⇔ |Δ PSNR| ≤ 0.003 dB (CPU vs GPU numerics); seed 0, Set12 + BSD68,
σ 15 / 25 / 50. Any mismatch voids the identical-draw pairing for that image.
Kaggle: CPU, internet on.

In [ ]:
import importlib.util, json, math, os, time, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

torch.set_num_threads(os.cpu_count())
SIGMAS = [15, 25, 50]
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
REF = json.loads(r'''{"Set12|02.png|15": 36.25797409566801, "Set12|01.png|15": 33.06775541363705, "Set12|04.png|15": 32.62310813615774, "Set12|03.png|15": 33.69287790602305, "Set12|05.png|15": 33.77317885463001, "Set12|06.png|15": 32.08758740607153, "Set12|07.png|15": 32.1742108962979, "Set12|08.png|15": 35.032109795950205, "Set12|09.png|15": 33.789107593382255, "Set12|10.png|15": 32.951801353632, "Set12|11.png|15": 32.670479411728266, "BSD68|test001.png|15": 28.52598748297184, "Set12|12.png|15": 32.871601649571225, "BSD68|test002.png|15": 32.967825618873164, "BSD68|test003.png|15": 33.00640903641652, "BSD68|test004.png|15": 33.574928247870275, "BSD68|test005.png|15": 30.35766951582641, "BSD68|test006.png|15": 35.513684902315894, "BSD68|test007.png|15": 31.563937573462223, "BSD68|test008.png|15": 29.155267638593635, "BSD68|test009.png|15": 31.809453716966047, "BSD68|test010.png|15": 32.380793210377384, "BSD68|test011.png|15": 32.46911851991009, "BSD68|test012.png|15": 31.216300484221208, "BSD68|test013.png|15": 32.23411613938703, "BSD68|test014.png|15": 33.654565636787886, "BSD68|test015.png|15": 30.054440148025968, "BSD68|test016.png|15": 31.486244332634755, "BSD68|test017.png|15": 35.298601826590456, "BSD68|test018.png|15": 35.75908282196835, "BSD68|test019.png|15": 31.01022539917075, "BSD68|test020.png|15": 31.80146464175945, "BSD68|test021.png|15": 29.65971112990621, "BSD68|test022.png|15": 30.177905034102604, "BSD68|test023.png|15": 32.195296870877264, "BSD68|test025.png|15": 32.42396518025992, "BSD68|test024.png|15": 31.61117011543701, "BSD68|test026.png|15": 31.176328018943703, "BSD68|test027.png|15": 32.70567344299797, "BSD68|test028.png|15": 35.71669611174572, "BSD68|test029.png|15": 27.69744162206628, "BSD68|test030.png|15": 32.62771633418533, "BSD68|test031.png|15": 27.835184951341265, "BSD68|test032.png|15": 29.54352552318988, "BSD68|test033.png|15": 30.92939622139931, "BSD68|test034.png|15": 35.79815738649537, "BSD68|test035.png|15": 30.79014610860891, "BSD68|test036.png|15": 29.36591882999707, "BSD68|test037.png|15": 30.9513783722438, "BSD68|test038.png|15": 31.979762097813627, "BSD68|test039.png|15": 37.33937088873785, "BSD68|test040.png|15": 32.03347802344564, "BSD68|test041.png|15": 30.783048937611536, "BSD68|test042.png|15": 32.46816772356328, "BSD68|test043.png|15": 33.5013295960334, "BSD68|test044.png|15": 31.42903498257596, "BSD68|test045.png|15": 36.49372724938788, "BSD68|test046.png|15": 29.336618461142407, "BSD68|test047.png|15": 29.453363287828203, "BSD68|test048.png|15": 33.457863592568195, "BSD68|test049.png|15": 33.64240840201968, "BSD68|test050.png|15": 29.545454306616257, "BSD68|test051.png|15": 31.492524888961505, "BSD68|test052.png|15": 34.311113014290015, "BSD68|test053.png|15": 33.39485597094372, "BSD68|test054.png|15": 32.075179353197335, "BSD68|test055.png|15": 30.44134770311629, "BSD68|test056.png|15": 27.077445918184928, "BSD68|test057.png|15": 30.94262514528814, "BSD68|test058.png|15": 32.81328525461795, "BSD68|test059.png|15": 32.52004277429904, "BSD68|test060.png|15": 33.31100788251487, "BSD68|test061.png|15": 32.81366723709586, "BSD68|test062.png|15": 37.24487638119874, "BSD68|test063.png|15": 29.546211720746584, "BSD68|test064.png|15": 29.150884392636016, "BSD68|test065.png|15": 31.57610755358279, "BSD68|test066.png|15": 41.36264042989068, "BSD68|test067.png|15": 27.732553735490008, "BSD68|test068.png|15": 29.284952427574463, "Set12|01.png|25": 30.74297843368109, "Set12|02.png|25": 34.22588506237104, "Set12|03.png|25": 31.3352005919238, "Set12|04.png|25": 30.074711850208967, "Set12|06.png|25": 29.47646960855156, "Set12|05.png|25": 31.032670310740045, "Set12|07.png|25": 29.76441016273011, "Set12|08.png|25": 33.015499953403314, "Set12|09.png|25": 31.50106095396461, "Set12|10.png|25": 30.78065524456971, "Set12|11.png|25": 30.34313888387581, "BSD68|test001.png|25": 26.082421690619807, "Set12|12.png|25": 30.627779167074763, "BSD68|test002.png|25": 30.49856049900667, "BSD68|test003.png|25": 30.65901380893584, "BSD68|test004.png|25": 30.99729483100421, "BSD68|test005.png|25": 27.784008280724063, "BSD68|test006.png|25": 33.393359922870786, "BSD68|test007.png|25": 28.985127232536925, "BSD68|test008.png|25": 26.389039942634565, "BSD68|test009.png|25": 29.33010420278487, "BSD68|test010.png|25": 29.94971815792939, "BSD68|test011.png|25": 29.82146349090697, "BSD68|test012.png|25": 28.683991142129123, "BSD68|test013.png|25": 29.818845128280174, "BSD68|test014.png|25": 31.30028840894372, "BSD68|test015.png|25": 27.41769196429946, "BSD68|test016.png|25": 28.715998054343377, "BSD68|test017.png|25": 33.31854945507179, "BSD68|test018.png|25": 33.56916048212299, "BSD68|test019.png|25": 28.519457873423075, "BSD68|test020.png|25": 29.197838818052094, "BSD68|test021.png|25": 26.684659597293884, "BSD68|test022.png|25": 27.59085921074962, "BSD68|test023.png|25": 29.35015531061812, "BSD68|test024.png|25": 28.93050291967522, "BSD68|test025.png|25": 29.75481162285589, "BSD68|test026.png|25": 28.79201029262266, "BSD68|test027.png|25": 30.344027852620563, "BSD68|test028.png|25": 33.35833299799254, "BSD68|test029.png|25": 24.619931151677896, "BSD68|test030.png|25": 30.161344876273382, "BSD68|test032.png|25": 26.83639094363645, "BSD68|test031.png|25": 24.954043601258583, "BSD68|test033.png|25": 28.19405595902389, "BSD68|test034.png|25": 34.09418446699209, "BSD68|test035.png|25": 28.393467779744206, "BSD68|test036.png|25": 27.76549789146943, "BSD68|test037.png|25": 28.83360473808021, "BSD68|test038.png|25": 29.486534097584386, "BSD68|test039.png|25": 34.77928484304864, "BSD68|test040.png|25": 29.823070400704754, "BSD68|test041.png|25": 28.338289734830894, "BSD68|test042.png|25": 30.07392196786247, "BSD68|test043.png|25": 30.845480440928803, "BSD68|test044.png|25": 28.81019066515958, "BSD68|test045.png|25": 34.444775427847404, "BSD68|test046.png|25": 26.679150574109578, "BSD68|test047.png|25": 26.813071338611188, "BSD68|test048.png|25": 30.61645255972778, "BSD68|test049.png|25": 31.80819016730052, "BSD68|test050.png|25": 27.10018870549915, "BSD68|test051.png|25": 29.09593919174586, "BSD68|test052.png|25": 32.47545636822543, "BSD68|test053.png|25": 31.52700190905582, "BSD68|test054.png|25": 29.610139543627216, "BSD68|test055.png|25": 27.854748400379677, "BSD68|test056.png|25": 24.19679283067643, "BSD68|test057.png|25": 28.52191482949337, "BSD68|test058.png|25": 31.12018538654466, "BSD68|test059.png|25": 30.659756459259825, "BSD68|test060.png|25": 31.280285886488286, "BSD68|test061.png|25": 30.50650349022625, "BSD68|test062.png|25": 35.004252225657886, "BSD68|test063.png|25": 26.748166443274226, "BSD68|test064.png|25": 26.87860930235448, "BSD68|test065.png|25": 29.154012898525444, "BSD68|test066.png|25": 39.08065787663083, "BSD68|test067.png|25": 24.897667114975803, "Set12|01.png|50": 27.84831931334069, "Set12|02.png|50": 31.528066054071022, "BSD68|test068.png|25": 26.3433007765146, "Set12|03.png|50": 28.02334514425752, "Set12|04.png|50": 26.65152668013932, "Set12|05.png|50": 27.342243367116986, "Set12|06.png|50": 26.180678333791803, "Set12|07.png|50": 26.916446455636223, "Set12|09.png|50": 28.33734460107287, "Set12|08.png|50": 30.203887515713987, "Set12|10.png|50": 27.82191980808657, "Set12|11.png|50": 27.51099149220209, "BSD68|test001.png|50": 23.34355285780713, "Set12|12.png|50": 27.63345134012283, "BSD68|test002.png|50": 27.636367473579213, "BSD68|test003.png|50": 27.50911416985085, "BSD68|test004.png|50": 27.89971616226072, "BSD68|test005.png|50": 24.97279969855972, "BSD68|test006.png|50": 30.83030535034557, "BSD68|test007.png|50": 25.830707846570547, "BSD68|test008.png|50": 23.2434017076846, "BSD68|test009.png|50": 26.39482396204842, "BSD68|test010.png|50": 27.09114304775285, "BSD68|test011.png|50": 26.531418498979168, "BSD68|test013.png|50": 27.030652498558304, "BSD68|test012.png|50": 25.580911159141912, "BSD68|test014.png|50": 28.362077559001754, "BSD68|test015.png|50": 24.395527382337864, "BSD68|test016.png|50": 25.006857572227357, "BSD68|test017.png|50": 30.938654329330998, "BSD68|test018.png|50": 30.91924311761153, "BSD68|test019.png|50": 25.82217911072172, "BSD68|test020.png|50": 26.1158193557608, "BSD68|test021.png|50": 23.185114686240595, "BSD68|test022.png|50": 24.42191691046384, "BSD68|test023.png|50": 25.7710069887364, "BSD68|test024.png|50": 25.52842340511463, "BSD68|test025.png|50": 26.611519624518778, "BSD68|test026.png|50": 26.003554013281093, "BSD68|test027.png|50": 27.48676908853553, "BSD68|test028.png|50": 30.53245860969476, "BSD68|test029.png|50": 21.17704181731531, "BSD68|test030.png|50": 27.24241453518632, "BSD68|test031.png|50": 21.63127328451365, "BSD68|test032.png|50": 23.68694857697276, "BSD68|test033.png|50": 25.15938057430786, "BSD68|test034.png|50": 31.577909961537507, "BSD68|test035.png|50": 25.71752899301714, "BSD68|test036.png|50": 26.52815335738918, "BSD68|test037.png|50": 26.54162935811561, "BSD68|test038.png|50": 26.51218290618108, "BSD68|test039.png|50": 30.818486824189826, "BSD68|test040.png|50": 26.953387500067578, "BSD68|test041.png|50": 25.4887301676712, "BSD68|test042.png|50": 27.01435987865747, "BSD68|test043.png|50": 27.42444336973528, "BSD68|test044.png|50": 25.61339608546464, "BSD68|test045.png|50": 32.03104474047145, "BSD68|test046.png|50": 23.441759237313484, "BSD68|test047.png|50": 23.805535956522345, "BSD68|test048.png|50": 26.879083043963952, "BSD68|test049.png|50": 29.707665463866103, "BSD68|test050.png|50": 24.437927236561407, "BSD68|test052.png|50": 30.30173858636768, "BSD68|test051.png|50": 25.907916244667312, "BSD68|test053.png|50": 29.45102321547938, "BSD68|test054.png|50": 26.35177542395204, "BSD68|test055.png|50": 24.79587790738628, "BSD68|test056.png|50": 21.22727115627051, "BSD68|test057.png|50": 26.012831648839025, "BSD68|test058.png|50": 29.354595708309915, "BSD68|test059.png|50": 28.61979456058697, "BSD68|test060.png|50": 29.06677713154408, "BSD68|test061.png|50": 27.915202559777995, "BSD68|test062.png|50": 31.702679333803555, "BSD68|test063.png|50": 23.56631467116071, "BSD68|test064.png|50": 24.40510184333225, "BSD68|test065.png|50": 26.37698667929532, "BSD68|test066.png|50": 35.75808263662815, "BSD68|test067.png|50": 21.645433142104213, "BSD68|test068.png|50": 23.10061935889811}''')   # "dataset|image|sigma" -> Restormer-gray PSNR in the lift job, seed 0
print(len(REF), "reference rows;", os.cpu_count(), "CPUs")


def download(url, path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        for attempt in range(5):
            try:
                urllib.request.urlretrieve(url, path)
                return path
            except Exception as e:
                print(f"  retry {attempt + 1} for {url}: {e}", flush=True)
                time.sleep(5 * (attempt + 1))
        raise RuntimeError(url)
    return path


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


GH = "https://github.com/{}/releases/download/{}/{}"
RAW = "https://raw.githubusercontent.com/cszn/DnCNN/master/testsets/{}/{}"
IMAGES = {"Set12": [f"{i:02d}.png" for i in range(1, 13)], "BSD68": [f"test{i:03d}.png" for i in range(1, 69)]}
src = download("https://raw.githubusercontent.com/swz30/Restormer/68dc6ac472db26f16361150cb7a96a1bc87da93f/basicsr/models/archs/restormer_arch.py",
               f"{CACHE}/restormer_arch.py")
RestormerNet = load_module("restormer_arch", src).Restormer
TEST = {ds: [(n, np.asarray(Image.open(download(RAW.format(ds, n), f"{CACHE}/{ds}_{n}")).convert("L"))) for n in ns]
        for ds, ns in IMAGES.items()}

In [ ]:
def noisy_of(clean_u8, sigma, seed):
    """nbGray.noisy_of, called single-threaded."""
    img = np.float32(clean_u8[..., None]) / 255.
    np.random.seed(seed=seed)
    img += np.random.normal(0, sigma / 255., img.shape)
    return torch.from_numpy(img[..., 0].copy())


def psnr(x, clean_u8):
    a = (x.clamp(0, 1) * 255).round().double()
    return 10 * math.log10(255 ** 2 / (a - torch.from_numpy(clean_u8).double()).pow(2).mean().item())


rows = []
t0 = time.time()
for s in SIGMAS:
    net = RestormerNet(inp_channels=1, out_channels=1, dim=48, num_blocks=[4, 6, 6, 8], num_refinement_blocks=4, heads=[1, 2, 4, 8],
                       ffn_expansion_factor=2.66, bias=False, LayerNorm_type="BiasFree", dual_pixel_task=False)
    w = download(GH.format("swz30/Restormer", "v1.0", f"gaussian_gray_denoising_sigma{s}.pth"), f"{CACHE}/restormer_gray_{s}.pth")
    net.load_state_dict(torch.load(w, map_location="cpu", weights_only=True)["params"], strict=True)
    net.eval().requires_grad_(False)
    for ds, items in TEST.items():
        for name, clean in items:
            y = noisy_of(clean, s, 0)
            h, wd = y.shape
            with torch.no_grad():
                out = net(F.pad(y[None, None], (0, (-wd) % 8, 0, (-h) % 8), mode="reflect"))[0, 0, :h, :wd].clamp(0, 1)
            p = psnr(out, clean)
            ref = REF[f"{ds}|{name}|{s}"]
            rows.append(dict(dataset=ds, image=name, sigma=s, psnr_cpu=p, psnr_lift_job=ref, delta=p - ref, noisy_psnr=psnr(y, clean)))
        print(f"σ {s} {ds}: done ({(time.time() - t0) / 60:.1f} min)", flush=True)

R = pd.DataFrame(rows)
R["same_draw"] = R.delta.abs() <= 0.003
R.to_csv(f"{OUT}/draw_check.csv", index=False)
S = R.groupby(["dataset", "sigma"]).agg(n=("delta", "size"), same=("same_draw", "sum"), max_abs_delta=("delta", lambda d: d.abs().max()))
print(S.to_string())
print("mismatching images:", R[~R.same_draw][["dataset", "image", "sigma", "delta"]].to_string(index=False))
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(all_same=bool(R.same_draw.all()), n_mismatch=int((~R.same_draw).sum()),
                   mismatches=R[~R.same_draw][["dataset", "image", "sigma", "delta"]].to_dict("records")), f, indent=1, default=float)